# Habituation with recovery and speedup (CVODE)



In [1]:
import os, sys, numpy as np

print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())


Python: 3.10.12
CWD: /local0/rossin/git/CRN-GenerativeAI/apps_nicolo/Habituation_3s_5r_MAK_REINFORCE


## 1) Import RL4CRN helpers


In [2]:
from typing import Any, Callable, Dict, List, Sequence, Tuple, Union
import numpy as np
from itertools import product

from RL4CRN.utils.input_interface import (
    register_task_kind,
    overrides_get,
    Configurator,
    TaskKindBase,
    TaskSpec,
)

from RL4CRN.utils.default_tasks.HabituationTaskKind import HabituationGapTaskKind # <-- Gap searches for 2nd and 3rd habituation hallmarks

## 2) Build a template IO/CRN


In [3]:
from RL4CRN.utils.crn_builders import build_simple_IOCRN

# choose preset
cfg = Configurator.preset("paper")

# select simulator and set tolerances
cfg.solver.algorithm = "CVODE"
cfg.solver.rtol = 1e-6
cfg.solver.atol = 1e-6

# build template IO/CRN
species_labels = ['X_1', 'X_2', 'X_3']
crn, species_labels = build_simple_IOCRN(
    species=species_labels,
    production_input_map={"X_1": "u_1"},
    degradation_input_map={},
    dilution_map={"X_1": 0.1, "X_2": 0.1, "X_3": 0.1},  # add dilution to ensure steady state exists
    production_map={"X_2": 0.1},  # add basal production to X_2 nonzero peaks
    output_species="X_3",
    solver=cfg.solver,
)

print("Template CRN built.")
print(" - num_inputs:", crn.num_inputs)
print(" - num_species:", len(species_labels))
print(" - species:", species_labels)


Template CRN built.
 - num_inputs: 1
 - num_species: 3
 - species: ['X_1', 'X_2', 'X_3']


## 3) Build the reaction library (MAK)


In [4]:
from RL4CRN.utils.library_builders import build_MAK_library

# library components
library_components = build_MAK_library(crn, species_labels, order=2)

library, M, K, masks = library_components
print("Library built.")
print(" - M (num reactions in library):", M)
print(" - K (num parameters in library):", K)


Library built.
 - M (num reactions in library): 91
 - K (num parameters in library): 91


## 4) Define the task


In [4]:
HabituationGapTaskKind.pretty_help()

### TaskKind `habituation_gap`

**Required params**
- `pulse_shapes`: List[(t_on, t_off)] OR a single (t_on, t_off)
- `gap_time`: float OFF gap duration
- `n_repeats_pre`: int pulses before gap
- `n_repeats_post`: int pulses after gap
- `u_values`: List[float] grid for u

**Optional params**
- `freq_weight`: float frequency penalty weight (default 1.0)
- `gap_weight`: float gap penalty weight (default 5.0)
- `recovery_tol`: float recovery tolerance (default 0.05)
- `dishabituate_rho`: float dishabituation constraint (default 1.0)
- `ratio_weights`: float or list (default 1.0)
- `min_peak`: float (default 0.1)
- `max_peak`: float (default 2.0)
- `n_t`: int samples per simulation (default task.n_t)
- `sensitization`: bool (default False)

**Notes**
- If pulse_shapes has one entry, cross-frequency slope penalties are disabled. If multiple shapes
  are provided, a monotonicity penalty encourages faster habituation at higher frequency.


In [ ]:
from RL4CRN.utils.input_interface import make_task, print_task_summary
import numpy as np

# Frequencies: 5s, 10s, 15s periods.
# Keep a fixed ON duration (e.g. 1s) and vary OFF so that (t_on + t_off) = period.
t_on = 1.0
periods = [5.0, 10.0, 15.0]                 # seconds
pulse_shapes = [(t_on, P - t_on) for P in periods]  # [(1,4), (1,9), (1,14)]

# IMPORTANT: Your multifreq loss sorts by period (smaller period = higher freq),
# so pass pulse_shapes in any order; it will be internally ordered.
# Still, it's good practice to provide them explicitly as above.

task = make_task(
    template_crn=crn,
    library_components=library_components,
    kind="habituation_gap",
    species_labels=species_labels,
    params={
        "pulse_shapes": pulse_shapes,   # <-- NEW (list of shapes)
        "n_repeats_pre": 10,
        "n_repeats_post": 10,
        "gap_time": 100.0,
        "n_t": 1000,
        "ic": "from_ss",
        "weights": "transient",
        "max_peak": 10.0,
        "min_peak": 0.1,
        "u_values": [1.0],
        "sensitization": False,

        # Multifreq-specific knobs (optional)
        "freq_weight": 1.0,        # weight of monotonic slope penalty across frequencies
        "gap_weight": 5.0,
        "recovery_tol": 0.05,
        "dishabituate_rho": 1.0,
        "ratio_weights": 1.0,      # or a list
    }
)

print_task_summary(task)

# --- Optional safety checks (recommended) ---
print("Sanity checks:")
print(" - template num_inputs:", crn.num_inputs)
print(" - first u shape:", np.asarray(task.u_list[0]).shape)
print(" - first u length:", len(task.u_list[0]))
assert len(task.u_list[0]) == crn.num_inputs, "Input dimension mismatch: u has wrong length!"

print("Pulse shapes used (t_on, t_off):", pulse_shapes)
print("Periods:", [a + b for a, b in pulse_shapes], " (smaller period = higher frequency)")


Task: habituation_gap
time_horizon: (1000,) [0..100.0]
num scenarios: 1
first 1 u: [array([1.], dtype=float32)]

Sanity checks:
 - template num_inputs: 1
 - first u shape: (1,)
 - first u length: 1
Pulse shapes used (t_on, t_off): [(1.0, 4.0), (1.0, 9.0), (1.0, 14.0)]
Periods: [5.0, 10.0, 15.0]  (smaller period = higher frequency)


## 5) Training configuration


In [ ]:
# ---- Train config ----
cfg.train.max_added_reactions = 5
cfg.train.epochs = 31
cfg.train.render_every = 5
cfg.train.seed = 0

Rendering options

In [8]:
cfg.render.n_best = 100
cfg.render.disregarded_percentage = 0.9
cfg.render.mode = {  # Mode of the experiment
    'style': 'logger', 
    'task': 'sensitization_gap', 
    'format': 'image',
    'topology': True
}

## 7) Create session + trainer


In [ ]:
import os
from datetime import datetime
from pytorch_lightning.loggers import CometLogger

task_name = "Habituation_h3_Task"
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Expect these in your environment:
#   COMET_API_KEY   (required)
#   COMET_WORKSPACE (required)
api_key = os.environ["COMET_API_KEY"]
workspace = os.environ["COMET_WORKSPACE"]

logger = CometLogger(
    api_key=api_key,
    project=task_name,
    workspace=workspace,
    name=f"{task_name}_{timestamp}",
)

logger = logger.experiment


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/redsnic/habituation-task/b9f0625e056746ca977b3b03b18eb1f0



In [11]:
from RL4CRN.utils.input_interface import make_session_and_trainer
trainer = make_session_and_trainer(cfg, task, logger=logger)

## 8) Train and save checkpoints


In [ ]:
# checkpoint_path = "habituation_task_chkpt.pkl"
checkpoint_path = f"{task_name}.pkl"
trainer.run(epochs=cfg.train.epochs, checkpoint_path=checkpoint_path)

trainer.save(checkpoint_path)

/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/utils/default_tasks/HabituationTaskKind.py:209: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  x_ss = fsolve(lambda x: crn.rate_function(0.0, x, u), x_prev)
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/utils/default_tasks/HabituationTaskKind.py:209: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last five Jacobian evaluations.
  x_ss = fsolve(lambda x: crn.rate_function(0.0, x, u), x_prev)
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/utils/default_tasks/HabituationTaskKind.py:209: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last five Jacobian evaluations.
  x_ss = fsolve(lambda x: crn.rate_function(0.0, x, u), x_prev)
/local0/rossin/git/CRN-GenerativeAI/.venv/


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails t

/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/utils/default_tasks/HabituationTaskKind.py:209: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  x_ss = fsolve(lambda x: crn.rate_function(0.0, x, u), x_prev)
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/utils/default_tasks/HabituationTaskKind.py:209: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last five Jacobian evaluations.
  x_ss = fsolve(lambda x: crn.rate_function(0.0, x, u), x_prev)
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/utils/default_tasks/HabituationTaskKind.py:209: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  x_ss = fsolve(lambda x: crn.rate_function(0.0, x, u), x_prev)
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.

[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.



[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails t

/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/utils/default_tasks/HabituationTaskKind.py:209: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  x_ss = fsolve(lambda x: crn.rate_function(0.0, x, u), x_prev)
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/utils/default_tasks/HabituationTaskKind.py:209: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  x_ss = fsolve(lambda x: crn.rate_function(0.0, x, u), x_prev)
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/utils/default_tasks/HabituationTaskKind.py:209: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  x_ss = fsolve(lambda x: crn.rate_function(0.0, x, u), x_prev)
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-pa

...


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails t

/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/utils/default_tasks/HabituationTaskKind.py:209: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  x_ss = fsolve(lambda x: crn.rate_function(0.0, x, u), x_prev)



[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails t

/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/environments/environment.py:506: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/environments/environment.py:506: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/environments/environment.py:506: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/environments/environment.py:506: UserWarning: The figure layout has changed to tight
  fig.tight_layout(rect=[0, 0, 1, 0.95])
/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/RL4CRN/environments/environment.py:506: UserWarning: The figure layout has changed to tight
  fig.tight_layout(re

Saved checkpoint: sensitization_task_chkpt.pkl

[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constraints.


[cvInitialSetup, Error: -22] y0 fails to satisfy constra